# 01 · Data Preparation and Leakage Controls

Defines the frozen modeling cohort, target, shared split, and the information boundary used throughout the portfolio.

## Data contract

The original raw-table build is documented in the team data-preparation work. This portfolio notebook starts from the frozen order-level master table rather than pretending the raw relational CSVs are committed here.

In [ ]:
from pathlib import Path
import pandas as pd

DATA_PATH = Path('../data/master_orders_clean_v2.csv')
if not DATA_PATH.exists():
    raise FileNotFoundError('Add the processed master table to data/master_orders_clean_v2.csv. See data/README.md.')

df = pd.read_csv(DATA_PATH, low_memory=False)
print(df.shape)
print(df['split'].value_counts())
print('negative-review rate:', round(df['review_bad'].mean(), 4))

In [ ]:
assert len(df) == 95824
assert df['order_id'].is_unique
assert df['review_bad'].isin([0,1]).all()
assert df['split'].value_counts().to_dict() == {'train': 76659, 'test': 19165}
print('Frozen cohort checks passed.')

## Target and feature timing

`review_bad` equals 1 for 1–2 star reviews and 0 for 3–5 stars. Review fields are never predictors. Delivery-stage variables are excluded from the placement model.

In [ ]:
TARGET_COLS=['review_score','review_bad']
REVIEW_TIMESTAMPS=['review_creation_date','review_answer_timestamp']
DELIVERY_ONLY=['delivery_time_days','delivery_vs_estimate_days','approval_delay_days','carrier_pickup_delay_days','shipping_transit_days','delivered_late','late_days','early_days']
print('Target fields:', TARGET_COLS)
print('Delivery-only fields:', DELIVERY_ONLY)